# Tier 1 — ANN generation forecasting

Run the configuration and preprocessing cells before the technology cells. The original chronological splits, scaling, 24-to-24 windows, architectures and technology-specific training settings are retained. `TRAIN_ANN=False` loads the supplied trained weights and runs the original forecasting/evaluation code. Set it to `True` to train. Cached ENTSO-E tables are used directly; only missing tables require `ENTSOE_API_KEY` and network access. New derived tables, forecasts and trained weights are written under `generated/upstream/tier1/`.


In [ ]:
from pathlib import Path
import os

# Open from the repository root or its notebooks/ subfolder.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the downloaded repository folder")
DATA = REPO_ROOT / "Csv files of input and output"
WORK = REPO_ROOT / "generated/upstream"
WORK.mkdir(parents=True, exist_ok=True)

TRAIN_ANN = False  # False: load the supplied .keras weights; True: retain original training.
ENTSOE_CACHE = DATA / "colab folders data to upload/data_entsoe_es_processed"
TIER1_WORK = WORK / "tier1"
TIER1_WORK.mkdir(parents=True, exist_ok=True)
TECH_ARCHIVES = {
    "gen_fossil_coal_oil_mw": "coaloil", "gen_biomass_mw": "biomass",
    "gen_wind_onshore_mw": "wind", "gen_fossil_gas_mw": "fossil gas",
    "gen_nuclear_mw": "nuclear", "gen_solar_mw": "solar", "gen_waste_mw": "waste",
    "gen_other_renewable_mw": "other renawables", "gen_hydro_run_of_river_mw": "hydro run of river",
    "gen_other_mw": "other", "gen_hydro_dispatchable_mw": "hydro dispatchable",
}


# 1. Tier 1 — ENTSO-E generation mix forecasting

The prep cell builds the hourly master feature tables, performs the chronological split, applies scaling, and creates the 24→24 windows used by the technology specific ANN models.


## 1.1 Prep block — fetch, process, split, scale, and create windows



In [ ]:
# =========================
# prep cell – ENTSO-E Fetch + Process + Split + Scale + Sliding Windows (2022–2024)
# =========================

import os, time, traceback, warnings
import numpy as np
import pandas as pd
from functools import reduce
from entsoe import EntsoePandasClient
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

# --------- CONFIG ----------
API_KEY = os.environ.get("ENTSOE_API_KEY", "PUT_YOUR_ENTSOE_API_KEY_HERE")
AREA          = "ES"                   # Spain
TZ_ENTSOE     = "Europe/Brussels"

YEARS_TRAIN   = [2022, 2023]
YEAR_TEST     = 2024

DATA_PROC_DIR = str(TIER1_WORK)

# path to MIBGAS day-ahead gas price Excel
MIBGAS_GAS_PATH = str(DATA / "colab folders data to upload/migbas/da_gas_price_migbasPVB_EURperMWh.xlsx")

VAL_DAYS      = 60   # last N days of 2023 as validation
TRAIN_WIN_H   = 24   # sliding window input hours
LABEL_WIN_H   = 24   # sliding window horizon hours

os.makedirs(DATA_PROC_DIR, exist_ok=True)

print("[INFO] Working dir:", os.getcwd())
print("[INFO] PROC  dir  :", DATA_PROC_DIR)

client = EntsoePandasClient(api_key=API_KEY)

# --------- COMMON HELPERS ----------

def _to_utc_hourly(df_like, tz_src=TZ_ENTSOE) -> pd.DataFrame:
    """
    Accepts Series or DataFrame from entsoe-py.
    Converts to hourly UTC DataFrame.
    DOES NOT drop all-NaN rows (we want to see missing hours).
    Resampling to 1H also fixes 15-minute data.
    """
    if df_like is None or len(df_like) == 0:
        return pd.DataFrame()

    # Series → DataFrame
    if isinstance(df_like, pd.Series):
        df = df_like.to_frame(name=df_like.name if df_like.name else "value")
    else:
        df = df_like.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

    # Localize to ENTSO-E market time, then convert to UTC
    if df.index.tz is None:
        df.index = df.index.tz_localize(tz_src, nonexistent="shift_forward", ambiguous="NaT")
    else:
        df = df.tz_convert(tz_src)
    df = df.tz_convert("UTC")

    # Cast numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Resample to hourly mean in UTC; keep NaN rows
    df = df.resample("1h").mean()

    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add calendar features based on datetime_utc index (UTC)."""
    df = df.copy()
    idx = df.index

    df["hour"] = idx.hour
    df["dow"] = idx.dayofweek
    df["month"] = idx.month

    doy = idx.dayofyear
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)

    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)

    # I did not add spain holidays
    df["is_holiday"] = 0

    return df


TECH_MAP = {
    "Biomass":                    "gen_biomass_mw",
    "Fossil Gas":                 "gen_fossil_gas_mw",
    "Fossil Hard coal":           "gen_fossil_hard_coal_mw",
    "Fossil Oil":                 "gen_fossil_oil_mw",
    "Hydro Pumped Storage":       "gen_hydro_pumped_storage_mw",
    "Hydro Run-of-river and poundage": "gen_hydro_run_of_river_mw",
    "Hydro Water Reservoir":      "gen_hydro_water_reservoir_mw",
    "Nuclear":                    "gen_nuclear_mw",
    "Solar":                      "gen_solar_mw",
    "Wind Onshore":               "gen_wind_onshore_mw",
    "Waste":                      "gen_waste_mw",
    "Other":                      "gen_other_mw",
    "Other renewable":            "gen_other_renewable_mw",
}

def _year_local_bounds(year: int):
    """
    Choose start/end in Europe/Brussels so that AFTER tz_convert to UTC and
    entsoe's internal reindexing, we cover exactly the UTC range
    [year-01-01 00:00, year-12-31 23:00].
    """
    start = pd.Timestamp(f"{year}-01-01 01:00", tz=TZ_ENTSOE)
    end   = pd.Timestamp(f"{year+1}-01-01 01:00", tz=TZ_ENTSOE)
    return start, end

def _iter_year_chunks(year: int, chunk_days: int = 30):
    """
    Iterate [start, end) windows in TZ_ENTSOE that cover the whole year,
    split into chunks of <= chunk_days.
    """
    start_year, end_year = _year_local_bounds(year)
    cur = start_year
    while cur < end_year:
        nxt = min(cur + pd.Timedelta(days=chunk_days), end_year)
        yield cur, nxt
        cur = nxt


# --------- CSV CACHE WRAPPER (per-year, per-signal) ----------

def cached_fetch_hourly(csv_path: str, fetch_fn, *args, **kwargs) -> pd.DataFrame:
    """
    If csv_path exists, load and return it (expects 'datetime_utc' column).
    Otherwise call fetch_fn(*args, **kwargs), save to csv_path (with datetime_utc),
    and return the DataFrame with datetime index.
    """
    supplied_cache = ENTSOE_CACHE / Path(csv_path).name
    if not os.path.exists(csv_path) and supplied_cache.exists():
        csv_path = str(supplied_cache)
    if os.path.exists(csv_path):
        print(f"[CACHE] Using existing file (no API call): {csv_path}")
        df = pd.read_csv(csv_path, parse_dates=["datetime_utc"])
        df = df.set_index("datetime_utc").sort_index()
        return df

    if API_KEY == "PUT_YOUR_ENTSOE_API_KEY_HERE":
        raise RuntimeError("A missing ENTSO-E cache requires the ENTSOE_API_KEY environment variable")
    print(f"[CACHE] No file at {csv_path} – calling ENTSO-E API once and saving.")
    df = fetch_fn(*args, **kwargs)
    if df is None or df.empty:
        print(f"[CACHE] WARNING: fetched empty df for {csv_path}")
        return df

    df = df.sort_index()
    df.index.name = "datetime_utc"
    df.reset_index().to_csv(csv_path, index=False)
    print(f"[CACHE] Saved fetched data -> {csv_path}")
    return df


# ---------LOAD MIBGAS DAY-AHEAD GAS PRICE & MAKE HOURLY ----------

def load_da_gas_price_mibgas(path: str = MIBGAS_GAS_PATH) -> pd.DataFrame:
    """
    Load daily MIBGAS PVB day-ahead gas price and expand to hourly UTC.

    Expected columns in Excel:
      - 'delivery_day'
      - 'da_gas_price_migbasPVB_EURperMWh'
    """
    if not os.path.exists(path):
        print(f"[GAS_DA] File not found at {path}. Skipping gas DA feature.")
        return pd.DataFrame()

    df = pd.read_excel(path)
    if "delivery_day" not in df.columns or "da_gas_price_migbasPVB_EURperMWh" not in df.columns:
        raise ValueError("[GAS_DA] Expected columns 'delivery_day' and "
                         "'da_gas_price_migbasPVB_EURperMWh' not found in MIBGAS file.")

    df["delivery_day"] = pd.to_datetime(df["delivery_day"])
    df = df.sort_values("delivery_day")

    # Daily series indexed by delivery_day
    daily = df.set_index("delivery_day")[["da_gas_price_migbasPVB_EURperMWh"]].copy()

    # Expand each day to 24 hourly rows with same price
    n_days = len(daily)
    if n_days == 0:
        print("[GAS_DA] Empty daily gas price data.")
        return pd.DataFrame()

    hourly_index = pd.date_range(
        start=daily.index.min(),
        end=daily.index.max() + pd.Timedelta(hours=23),
        freq="1H",
        tz="UTC",
    )
    hourly = daily.loc[daily.index.repeat(24)].copy()
    hourly.index = hourly_index
    hourly.index.name = "datetime_utc"

    print(f"[GAS_DA] Loaded MIBGAS gas DA prices: daily_rows={n_days}, hourly_rows={len(hourly)}, "
          f"range={hourly.index.min()} -> {hourly.index.max()}")

    # Save hourly gas DA feature as CSV for inspection
    gas_csv_path = os.path.join(DATA_PROC_DIR, "da_gas_price_migbasPVB_EURperMWh_hourly.csv")
    hourly.reset_index().to_csv(gas_csv_path, index=False)
    print(f"[GAS_DA] Saved hourly gas DA price -> {gas_csv_path}")

    return hourly


# --------- FETCHERS ----------
# the entso-e platform updated early 2026, so you could easy download the csv files directly, and this code might have issues in fetching correct timeframes, please be aware
def fetch_gen_actual(country_code: str, year: int) -> pd.DataFrame:
    """
    Actual Generation per Production Type → hourly UTC, only selected techs.
    Fetched in 30-day chunks.
    """
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_generation(country_code, start=start, end=end, psr_type=None)
        except Exception as e:
            print(f"[GEN_ACT] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[GEN_ACT] EMPTY chunk {country_code} {year} {start} -> {end}")
            continue

        df = raw_part.copy()

        # Flatten multiindex and keep "Actual Aggregated" if present
        if isinstance(df.columns, pd.MultiIndex):
            agg_level = None
            for lvl in range(df.columns.nlevels):
                vals = set(map(str, df.columns.get_level_values(lvl)))
                if "Actual Aggregated" in vals or "Actual Consumption" in vals:
                    agg_level = lvl
                    break

            if agg_level is not None:
                cols_keep = [c for c in df.columns if str(c[agg_level]) == "Actual Aggregated"]
                df = df[cols_keep]
                tech_levels = [i for i in range(df.columns.nlevels) if i != agg_level]
                df = df.T.groupby(level=tech_levels[-1]).sum(min_count=1).T
            else:
                df = df.T.groupby(level=-1).sum(min_count=1).T

        # Keep only technologies we care about, rename to gen_* cols
        df.columns = [str(c).strip() for c in df.columns]
        keep_cols = {k: v for k, v in TECH_MAP.items() if k in df.columns}
        if not keep_cols:
            print(f"[GEN_ACT] No mapped techs found in chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df = df[list(keep_cols.keys())]
        df = df.rename(columns=keep_cols)

        df_part = _to_utc_hourly(df)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[GEN_ACT] EMPTY for {country_code} {year} (no valid chunks)")
        return pd.DataFrame()

    df_all = pd.concat(dfs, axis=0).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep="first")]

    # Clip strictly to UTC year bounds
    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df_all = df_all.loc[(df_all.index >= utc_start) & (df_all.index < utc_end)]

    print(f"[GEN_ACT] {year}: shape={df_all.shape}, cols={list(df_all.columns)}, "
          f"range={df_all.index.min()} -> {df_all.index.max()}")
    return df_all


def fetch_load_da(country_code: str, year: int) -> pd.DataFrame:
    """Day Ahead Total Load Forecast (UTC), fetched in 30-day chunks."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_load_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[LOAD_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[LOAD_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[LOAD_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    if df.shape[1] > 1:
        df = df.iloc[:, :1]
    df.columns = ["da_total_load_mw"]

    print(f"[LOAD_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def fetch_gen_da_total(country_code: str, year: int) -> pd.DataFrame:
    """Day-ahead total generation forecast (ENTSOE returns total, not per tech)."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_generation_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[GEN_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[GEN_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        if isinstance(raw_part, pd.Series):
            df_tmp = raw_part.to_frame("da_gen_total_mw")
        else:
            df_tmp = raw_part.sum(axis=1).to_frame("da_gen_total_mw")

        df_part = _to_utc_hourly(df_tmp)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[GEN_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    print(f"[GEN_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def fetch_wind_solar_da(country_code: str, year: int) -> pd.DataFrame:
    """Day-ahead forecast for wind + solar (total) in UTC, 30-day chunks."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_wind_and_solar_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[WS_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[WS_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(raw_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[WS_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    cols = list(df.columns)
    wind_cols  = [c for c in cols if "Wind"  in str(c)]
    solar_cols = [c for c in cols if "Solar" in str(c)]

    out = pd.DataFrame(index=df.index)
    if wind_cols:
        out["da_wind_total_mw"] = df[wind_cols].sum(axis=1)
    if solar_cols:
        out["da_solar_total_mw"] = df[solar_cols].sum(axis=1)

    print(f"[WS_DA] {year}: shape={out.shape}, cols={list(out.columns)}, "
          f"range={out.index.min()} -> {out.index.max()}")
    return out


def fetch_exchanges_da(country_from: str, country_to: str, year: int) -> pd.DataFrame:
    """
    Day-ahead scheduled commercial exchanges between ES and neighbor, in UTC.
    Positive series is export from 'country_from' to 'country_to'.
    """
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_scheduled_exchanges(country_from, country_to, start=start, end=end)
        except Exception as e:
            print(f"[XCH_DA] Error {country_from}-{country_to} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[XCH_DA] EMPTY chunk {country_from}-{country_to} {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        if df_part.shape[1] > 1:
            df_part = df_part.iloc[:, :1]
        df_part.columns = ["flow_mw"]

        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[XCH_DA] EMPTY for {country_from}-{country_to} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    flow = df["flow_mw"]
    imports = (-flow).clip(lower=0)
    exports = ( flow).clip(lower=0)

    if country_to == "FR":
        out = pd.DataFrame({
            "da_import_fr_mw": imports,
            "da_export_fr_mw": exports,
        }, index=df.index)
    elif country_to == "PT":
        out = pd.DataFrame({
            "da_import_pt_mw": imports,
            "da_export_pt_mw": exports,
        }, index=df.index)
    else:
        out = pd.DataFrame({
            f"da_import_{country_to.lower()}_mw": imports,
            f"da_export_{country_to.lower()}_mw": exports,
        }, index=df.index)

    print(f"[XCH_DA] {country_from}-{country_to} {year}: shape={out.shape}, "
          f"range={out.index.min()} -> {out.index.max()}")
    return out


# ENTSO-E Day-Ahead Electricity Price
def fetch_elec_price_da(country_code: str, year: int) -> pd.DataFrame:
    """
    Day-ahead electricity price (EUR/MWh) from ENTSO-E, hourly UTC.
    Uses 'Without Sequence - Day-ahead' prices.
    """
    dfs = []
    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_day_ahead_prices(country_code, start=start, end=end)
        except Exception as e:
            print(f"[PRICE_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[PRICE_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[PRICE_DA] EMPTY for {country_code} {year} (no valid chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    if df.shape[1] > 1:
        df = df.iloc[:, :1]
    df.columns = ["da_elec_price_EURperMWh"]

    print(f"[PRICE_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def merge_year_and_report_missing(dfs_dict, year: int, outdir_processed: str) -> pd.DataFrame:
    """
    Merge all pieces for a given year on UTC hourly index, do DA interpolation,
    build hydro_dispatchable, coal+oil aggregate, DA-derived features, calendar features.
    NO lag/ramp. NO row drops for missing gen_*.
    """
    # Align on datetime_utc index
    for name, df in dfs_dict.items():
        if df is None or df.empty:
            print(f"[MERGE] Warning: {name} empty for {year}")
            continue
        if "datetime_utc" in df.columns:
            df = df.set_index("datetime_utc")
        df = df.sort_index()
        dfs_dict[name] = df

    frames = [df for df in dfs_dict.values() if df is not None and not df.empty]
    if not frames:
        raise RuntimeError(f"[MERGE] No frames for year {year}")

    master = reduce(lambda left, right: left.join(right, how="outer"), frames)
    master.index.name = "datetime_utc"
    master = master.sort_index()

    print(f"\n[MERGE] Year {year}: merged shape BEFORE interpolation = {master.shape}")
    print(f"[MERGE] Time range: {master.index.min()} -> {master.index.max()}")

    # Missingness BEFORE interpolation
    miss_mask = master.isna()

    long_missing = miss_mask.stack().reset_index()
    long_missing.columns = ["datetime_utc", "column", "is_missing"]
    long_missing = long_missing[long_missing["is_missing"]]

    miss_detail_path = os.path.join(outdir_processed, f"missing_hours_ES_{year}.csv")
    long_missing.to_csv(miss_detail_path, index=False)
    print(f"[MISS] Saved detailed missing hours -> {miss_detail_path} (rows={len(long_missing)})")

    miss_summary = miss_mask.sum().reset_index()
    miss_summary.columns = ["column", "n_missing"]
    miss_summary["year"] = year
    miss_summary_path = os.path.join(outdir_processed, f"missing_summary_ES_{year}.csv")
    miss_summary.to_csv(miss_summary_path, index=False)
    print(f"[MISS] Saved missing summary -> {miss_summary_path}")

    # Interpolate DA features (forecasts only) – first pass
    da_cols = [c for c in master.columns if c.startswith("da_")]
    print(f"[INTERP] Interpolating DA columns (year-level): {da_cols}")
    master[da_cols] = master[da_cols].interpolate(method="time", limit_direction="both")

    # ----- Hydro dispatchable aggregate: reservoir + pumped -----
    has_res = "gen_hydro_water_reservoir_mw" in master.columns
    has_pst = "gen_hydro_pumped_storage_mw" in master.columns
    if has_res or has_pst:
        res_series = master["gen_hydro_water_reservoir_mw"].fillna(0.0) if has_res else 0.0
        pst_series = master["gen_hydro_pumped_storage_mw"].fillna(0.0) if has_pst else 0.0
        master["gen_hydro_dispatchable_mw"] = res_series + pst_series
        print(f"[HYDRO] Added gen_hydro_dispatchable_mw for year {year}")

    # ----- Fossil coal + oil aggregate (NaNs treated as 0 for the sum) -----
    has_coal = "gen_fossil_hard_coal_mw" in master.columns
    has_oil  = "gen_fossil_oil_mw" in master.columns
    if has_coal or has_oil:
        coal_series = master["gen_fossil_hard_coal_mw"].fillna(0.0) if has_coal else 0.0
        oil_series  = master["gen_fossil_oil_mw"].fillna(0.0) if has_oil else 0.0
        master["gen_fossil_coal_oil_mw"] = coal_series + oil_series
        print(f"[FOSSIL] Added gen_fossil_coal_oil_mw for year {year}")

    # Derived DA features (now that DA + hydro are there)
    eps = 1e-6
    if "da_total_load_mw" in master.columns:
        master["da_residual_load_mw"] = (
            master["da_total_load_mw"]
            - master.get("da_wind_total_mw", 0.0)
            - master.get("da_solar_total_mw", 0.0)
        )

        master["da_res_share_wind"] = (
            master.get("da_wind_total_mw", 0.0) / (master["da_total_load_mw"] + eps)
        )
        master["da_res_share_solar"] = (
            master.get("da_solar_total_mw", 0.0) / (master["da_total_load_mw"] + eps)
        )

    master["da_net_imports_mw"] = (
        master.get("da_import_fr_mw", 0.0)
        + master.get("da_import_pt_mw", 0.0)
        - master.get("da_export_fr_mw", 0.0)
        - master.get("da_export_pt_mw", 0.0)
    )
    if "da_total_load_mw" in master.columns:
        master["da_import_share"] = master["da_net_imports_mw"] / (master["da_total_load_mw"] + eps)

    # Calendar features (UTC)
    master = master.sort_index()
    master = add_calendar_features(master)

    # Save raw per-year master (with NaNs still present in non-DA columns)
    master_path = os.path.join(outdir_processed, f"master_ES_{year}.csv")
    master.reset_index().to_csv(master_path, index=False)
    print(f"[MASTER] Saved master ES {year} -> {master_path} (rows={len(master)}, cols={len(master.columns)})")

    return master


# --------- PRECOMPUTE GAS DA HOURLY SERIES ----------
gas_da_hourly = load_da_gas_price_mibgas()

# --------- BUILD MASTERS FOR TRAIN YEARS (2022–2023) ----------
masters_train = []

for yr in YEARS_TRAIN:
    try:
        print(f"\n========== YEAR {yr} ==========")

        gen_actual_path   = os.path.join(DATA_PROC_DIR, f"gen_actual_{AREA}_{yr}.csv")
        load_da_path      = os.path.join(DATA_PROC_DIR, f"da_total_load_{AREA}_{yr}.csv")
        gen_da_total_path = os.path.join(DATA_PROC_DIR, f"da_gen_total_{AREA}_{yr}.csv")
        ws_da_path        = os.path.join(DATA_PROC_DIR, f"da_wind_solar_{AREA}_{yr}.csv")
        exch_es_fr_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_FR_{yr}.csv")
        exch_es_pt_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_PT_{yr}.csv")
        price_da_path     = os.path.join(DATA_PROC_DIR, f"da_elec_price_{AREA}_{yr}.csv")

        gen_actual    = cached_fetch_hourly(gen_actual_path,   fetch_gen_actual, AREA, yr)
        load_da       = cached_fetch_hourly(load_da_path,      fetch_load_da, AREA, yr)
        gen_da_total  = cached_fetch_hourly(gen_da_total_path, fetch_gen_da_total, AREA, yr)
        ws_da         = cached_fetch_hourly(ws_da_path,        fetch_wind_solar_da, AREA, yr)
        exch_es_fr    = cached_fetch_hourly(exch_es_fr_path,   fetch_exchanges_da, "ES", "FR", yr)
        exch_es_pt    = cached_fetch_hourly(exch_es_pt_path,   fetch_exchanges_da, "ES", "PT", yr)
        price_da      = cached_fetch_hourly(price_da_path,     fetch_elec_price_da, AREA, yr)

        # Subset gas DA hourly to this year (if available)
        if gas_da_hourly is not None and not gas_da_hourly.empty:
            mask_gas = (
                (gas_da_hourly.index >= pd.Timestamp(f"{yr}-01-01 00:00", tz="UTC")) &
                (gas_da_hourly.index <  pd.Timestamp(f"{yr+1}-01-01 00:00", tz="UTC"))
            )
            gas_da_year = gas_da_hourly.loc[mask_gas].copy()
        else:
            gas_da_year = pd.DataFrame()
            print(f"[GAS_DA] No gas DA data available for year {yr}")

        dfs_year = {
            "gen_actual": gen_actual,
            "load_da": load_da,
            "gen_da_total": gen_da_total,
            "ws_da": ws_da,
            "xch_fr": exch_es_fr,
            "xch_pt": exch_es_pt,
            "price_da": price_da,
            "gas_da": gas_da_year,
        }

        master_y = merge_year_and_report_missing(dfs_year, yr, DATA_PROC_DIR)
        masters_train.append(master_y)

    except Exception as e:
        print(f"[ERROR] Year {yr}: {e}")
        traceback.print_exc()
    time.sleep(1.0)

if not masters_train:
    raise RuntimeError("[TRAIN] No master data built for TRAIN years.")

master_train = pd.concat(masters_train, axis=0).sort_index()
print("\n[TRAIN] CONCAT 2022–2023:", master_train.shape,
      "| range:", master_train.index.min(), "->", master_train.index.max())

train_master_path = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_raw.csv")
master_train.reset_index().to_csv(train_master_path, index=False)
print(f"[TRAIN] Saved raw concatenated TRAIN master -> {train_master_path}")

# --------- BUILD MASTER FOR TEST YEAR (2024) ----------
print(f"\n========== YEAR {YEAR_TEST} (TEST) ==========")
try:
    gen_actual_path   = os.path.join(DATA_PROC_DIR, f"gen_actual_{AREA}_{YEAR_TEST}.csv")
    load_da_path      = os.path.join(DATA_PROC_DIR, f"da_total_load_{AREA}_{YEAR_TEST}.csv")
    gen_da_total_path = os.path.join(DATA_PROC_DIR, f"da_gen_total_{AREA}_{YEAR_TEST}.csv")
    ws_da_path        = os.path.join(DATA_PROC_DIR, f"da_wind_solar_{AREA}_{YEAR_TEST}.csv")
    exch_es_fr_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_FR_{YEAR_TEST}.csv")
    exch_es_pt_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_PT_{YEAR_TEST}.csv")
    price_da_path     = os.path.join(DATA_PROC_DIR, f"da_elec_price_{AREA}_{YEAR_TEST}.csv")

    gen_actual    = cached_fetch_hourly(gen_actual_path,   fetch_gen_actual, AREA, YEAR_TEST)
    load_da       = cached_fetch_hourly(load_da_path,      fetch_load_da, AREA, YEAR_TEST)
    gen_da_total  = cached_fetch_hourly(gen_da_total_path, fetch_gen_da_total, AREA, YEAR_TEST)
    ws_da         = cached_fetch_hourly(ws_da_path,        fetch_wind_solar_da, AREA, YEAR_TEST)
    exch_es_fr    = cached_fetch_hourly(exch_es_fr_path,   fetch_exchanges_da, "ES", "FR", YEAR_TEST)
    exch_es_pt    = cached_fetch_hourly(exch_es_pt_path,   fetch_exchanges_da, "ES", "PT", YEAR_TEST)
    price_da      = cached_fetch_hourly(price_da_path,     fetch_elec_price_da, AREA, YEAR_TEST)

    if gas_da_hourly is not None and not gas_da_hourly.empty:
        mask_gas = (
            (gas_da_hourly.index >= pd.Timestamp(f"{YEAR_TEST}-01-01 00:00", tz="UTC")) &
            (gas_da_hourly.index <  pd.Timestamp(f"{YEAR_TEST+1}-01-01 00:00", tz="UTC"))
        )
        gas_da_year = gas_da_hourly.loc[mask_gas].copy()
    else:
        gas_da_year = pd.DataFrame()
        print(f"[GAS_DA] No gas DA data available for year {YEAR_TEST}")

    dfs_year = {
        "gen_actual": gen_actual,
        "load_da": load_da,
        "gen_da_total": gen_da_total,
        "ws_da": ws_da,
        "xch_fr": exch_es_fr,
        "xch_pt": exch_es_pt,
        "price_da": price_da,
        "gas_da": gas_da_year,
    }

    master_test = merge_year_and_report_missing(dfs_year, YEAR_TEST, DATA_PROC_DIR)
    master_test = master_test.sort_index()

except Exception as e:
    print(f"[ERROR] Year {YEAR_TEST}: {e}")
    traceback.print_exc()
    raise

test_master_path = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_raw.csv")
master_test.reset_index().to_csv(test_master_path, index=False)
print(f"[TEST] Saved raw TEST master -> {test_master_path}")
print("[TEST] master_test:", master_test.shape,
      "| range:", master_test.index.min(), "->", master_test.index.max())

# --------- GLOBAL LINEAR INTERPOLATION (ALL NUMERIC COLS, TRAIN & TEST) ----------
def interpolate_all_numeric(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.sort_index()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"[INTERP-ALL] {name}: numeric cols={len(num_cols)}")
    df[num_cols] = df[num_cols].interpolate(method="time", limit_direction="both")
    return df

master_train = interpolate_all_numeric(master_train, "TRAIN(2022–2023)")
master_test  = interpolate_all_numeric(master_test,  "TEST(2024)")

train_interp_path = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_interp.csv")
test_interp_path  = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_interp.csv")
master_train.reset_index().to_csv(train_interp_path, index=False)
master_test.reset_index().to_csv(test_interp_path, index=False)
print(f"[SAVE] TRAIN interpolated -> {train_interp_path}")
print(f"[SAVE] TEST  interpolated -> {test_interp_path}")

# --------- FEATURE SELECTION (GEN, DA, CALENDAR) ----------
gen_cols = [c for c in master_train.columns if c.startswith("gen_")]
da_cols  = [c for c in master_train.columns if c.startswith("da_")]
cal_cols = ["hour", "dow", "month", "hour_sin", "hour_cos",
            "doy_sin", "doy_cos", "is_weekend", "is_holiday"]
cal_cols = [c for c in cal_cols if c in master_train.columns]

# --- AGGREGATE CLEANUP FOR TARGET TECHS ---
# If we have dispatchable hydro, drop component techs from the prediction list
if "gen_hydro_dispatchable_mw" in gen_cols:
    for c in ["gen_hydro_water_reservoir_mw", "gen_hydro_pumped_storage_mw"]:
        if c in gen_cols:
            gen_cols.remove(c)

# Ensure coal+oil aggregate is included and donors removed
if "gen_fossil_coal_oil_mw" in master_train.columns:
    if "gen_fossil_coal_oil_mw" not in gen_cols:
        gen_cols.append("gen_fossil_coal_oil_mw")
    for c in ["gen_fossil_hard_coal_mw", "gen_fossil_oil_mw"]:
        if c in gen_cols:
            gen_cols.remove(c)

print("\n[FEATURES] gen_cols:", gen_cols)
print("[FEATURES] da_cols :", da_cols)
print("[FEATURES] cal_cols:", cal_cols)

# Save feature lists
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list = (
    pd.concat([
        pd.DataFrame({"type": "gen", "name": gen_cols}),
        pd.DataFrame({"type": "da",  "name": da_cols}),
        pd.DataFrame({"type": "cal", "name": cal_cols}),
    ], axis=0, ignore_index=True)
)
feat_df_list.to_csv(feat_list_path, index=False)
print(f"[FEATURES] Saved feature lists -> {feat_list_path}")

# --------- TRAIN / VAL SPLIT (TIME-BASED INSIDE 2022–2023) ----------
train_end = master_train.index.max()
val_start = train_end - pd.Timedelta(days=VAL_DAYS) + pd.Timedelta(hours=1)

train_df = master_train.loc[master_train.index < val_start].copy()
val_df   = master_train.loc[master_train.index >= val_start].copy()

print("\n[SPLIT] TRAIN_df:", train_df.shape,
      "| range:", train_df.index.min(), "->", train_df.index.max())
print("[SPLIT] VAL_df  :", val_df.shape,
      "| range:", val_df.index.min(), "->", val_df.index.max())

# --------- SCALE (MINMAX) – FIT ON TRAIN ONLY ----------
# For MLPs, we keep:
#   - history features: gen_cols + cal_cols (known from past)
#   - DA features: da_cols (future horizon)
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols

all_feature_cols  = hist_feature_cols + da_feature_cols

# ensure no duplicates in all_feature_cols
all_feature_cols = list(dict.fromkeys(all_feature_cols))

# consistency check – must exist in all dfs
for c in all_feature_cols:
    if c not in train_df.columns or c not in val_df.columns or c not in master_test.columns:
        raise ValueError(f"[FEATURES] Column {c} not present in train/val/test dfs.")

# We also keep all gen_cols in y, but per-tech training will pick one target later.
# For scaling we treat all feature columns with MinMax, and also all gen_* as potential targets for later.
X_train_raw = train_df[all_feature_cols].values
X_val_raw   = val_df[all_feature_cols].values
X_test_raw  = master_test[all_feature_cols].values

y_train_raw = train_df[gen_cols].values
y_val_raw   = val_df[gen_cols].values
y_test_raw  = master_test[gen_cols].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_val_scaled   = scaler_X.transform(X_val_raw)
X_test_scaled  = scaler_X.transform(X_test_raw)

y_train_scaled = scaler_y.fit_transform(y_train_raw)
y_val_scaled   = scaler_y.transform(y_val_raw)
y_test_scaled  = scaler_y.transform(y_test_raw)

train_scaled = pd.DataFrame(X_train_scaled, index=train_df.index, columns=all_feature_cols)
val_scaled   = pd.DataFrame(X_val_scaled,   index=val_df.index,   columns=all_feature_cols)
test_scaled  = pd.DataFrame(X_test_scaled,  index=master_test.index, columns=all_feature_cols)

# add scaled gen_* columns (targets) back
for i, col in enumerate(gen_cols):
    train_scaled[col] = y_train_scaled[:, i]
    val_scaled[col]   = y_val_scaled[:, i]
    test_scaled[col]  = y_test_scaled[:, i]

print("\n[SCALE] train_scaled:", train_scaled.shape,
      "| range:", train_scaled.index.min(), "->", train_scaled.index.max())
print("[SCALE] val_scaled  :", val_scaled.shape,
      "| range:", val_scaled.index.min(),   "->", val_scaled.index.max())
print("[SCALE] test_scaled :", test_scaled.shape,
      "| range:", test_scaled.index.min(),  "->", test_scaled.index.max())

train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

train_scaled.reset_index().to_csv(train_scaled_path, index=False)
val_scaled.reset_index().to_csv(val_scaled_path,   index=False)
test_scaled.reset_index().to_csv(test_scaled_path, index=False)

print(f"[SAVE] train_scaled_hourly -> {train_scaled_path}")
print(f"[SAVE] val_scaled_hourly   -> {val_scaled_path}")
print(f"[SAVE] test_scaled_hourly  -> {test_scaled_path}")

# --------- SLIDING WINDOWS 24→24 (CarbonCast-style, with DA horizon)
def build_sliding_windows_with_da(df_scaled: pd.DataFrame,
                                  hist_cols,
                                  da_cols,
                                  gen_cols_for_y,
                                  input_hours=24,
                                  label_hours=24):
    """
    Building CarbonCast-style sliding windows:

    For each index i:
      H = df[hist_cols][i : i+input_hours]                  (past 24h)
      F = df[da_cols][i+input_hours : i+input_hours+label_hours] (DA horizon 24h)
      X_window = concat(H_t, F_t) along feature axis       -> shape (24, F_hist+F_da)
      y_window = df[gen_cols_for_y][i+input_hours : i+input_hours+label_hours]

    Returned X is flattened for MLP: (N_samples, 24 * (F_hist+F_da))
    Returned y is shape: (N_samples, 24, n_gen_cols_for_y)  – you will pick per-tech later.
    """
    df_scaled = df_scaled.sort_index()
    H_data = df_scaled[hist_cols].values
    F_data = df_scaled[da_cols].values
    Y_data = df_scaled[gen_cols_for_y].values

    n = len(df_scaled)
    max_i = n - (input_hours + label_hours) + 1
    if max_i <= 0:
        raise RuntimeError("[SLIDING] Not enough data points for given input/label hours.")

    X_list, Y_list = [], []

    for i in range(max_i):
        h_block = H_data[i : i + input_hours, :]  # (24, F_hist)
        f_block = F_data[i + input_hours : i + input_hours + label_hours, :]  # (24, F_da)

        if h_block.shape[0] != input_hours or f_block.shape[0] != label_hours:
            continue  # safety

        # concatenate along feature axis: shape (24, F_hist + F_da)
        x_block = np.concatenate([h_block, f_block], axis=1)
        y_block = Y_data[i + input_hours : i + input_hours + label_hours, :]  # (24, n_gen_cols_for_y)

        X_list.append(x_block.reshape(-1))  # flatten to (24*(F_hist+F_da),)
        Y_list.append(y_block)              # keep (24, n_gen_cols_for_y)

    X = np.stack(X_list)
    Y = np.stack(Y_list)

    return X, Y

print("\n[SLIDING] Building sliding windows 24→24 for TRAIN and VAL...")

X_train, Y_train = build_sliding_windows_with_da(
    train_scaled, hist_feature_cols, da_feature_cols, gen_cols,
    input_hours=TRAIN_WIN_H, label_hours=LABEL_WIN_H
)

X_val, Y_val = build_sliding_windows_with_da(
    val_scaled, hist_feature_cols, da_feature_cols, gen_cols,
    input_hours=TRAIN_WIN_H, label_hours=LABEL_WIN_H
)

print("[SLIDING] X_train:", X_train.shape, "| Y_train:", Y_train.shape)
print("[SLIDING] X_val  :", X_val.shape,   "| Y_val  :", Y_val.shape)

# Save sliding-window pairs as CSV (flattened X, flattened Y per sample)
# Y has shape (N, 24, n_gen_cols)
n_train, _, n_gen_targets = Y_train.shape
n_val,   _, _             = Y_val.shape

Y_train_flat = Y_train.reshape(n_train, LABEL_WIN_H * n_gen_targets)
Y_val_flat   = Y_val.reshape(n_val,   LABEL_WIN_H * n_gen_targets)

X_train_cols = [f"x_{i}" for i in range(X_train.shape[1])]
X_val_cols   = [f"x_{i}" for i in range(X_val.shape[1])]

Y_train_cols = [f"y_{tech}_{h}"
                for tech in gen_cols
                for h in range(LABEL_WIN_H)]
Y_val_cols   = [f"y_{tech}_{h}"
                for tech in gen_cols
                for h in range(LABEL_WIN_H)]

X_train_df = pd.DataFrame(X_train, columns=X_train_cols)
X_val_df   = pd.DataFrame(X_val,   columns=X_val_cols)
Y_train_df = pd.DataFrame(Y_train_flat, columns=Y_train_cols)
Y_val_df   = pd.DataFrame(Y_val_flat,   columns=Y_val_cols)

X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_df.to_csv(X_train_path, index=False)
Y_train_df.to_csv(Y_train_path, index=False)
X_val_df.to_csv(X_val_path,     index=False)
Y_val_df.to_csv(Y_val_path,     index=False)

print(f"[SAVE] X_train_sliding -> {X_train_path}")
print(f"[SAVE] Y_train_sliding -> {Y_train_path}")
print(f"[SAVE] X_val_sliding   -> {X_val_path}")
print(f"[SAVE] Y_val_sliding   -> {Y_val_path}")

# --------- INITIAL HISTORY FOR FORECASTING (END OF VAL) ----------
# Last 24 hours of VAL (scaled) – full row with all features + gen_*.
history_init = val_scaled.iloc[-TRAIN_WIN_H:].copy()
history_init_path = os.path.join(DATA_PROC_DIR, "entsoe_history_init_val_last24.csv")
history_init.reset_index().to_csv(history_init_path, index=False)
print(f"[SAVE] history_init (last 24h of VAL) -> {history_init_path}")

print("\n[DONE] Prep cell finished. You can now build per-tech MLPs using:")
print("- Sliding window TRAIN/VAL CSVs for supervised learning")
print("- hourly scaled TRAIN/VAL/TEST for day-ahead walk-forward forecasting")
print("- gen_cols / da_cols / cal_cols from feature_lists_entsoe.csv")


## 1.2 Technology specific ANN training and 2024 testing

Each technology is kept as a separate code block because the final manuscript/SI used technology specific ANN configurations rather than one shared architecture.  
Excluded here: individual hard coal, fossil oil, hydro pumped storage, and hydro water reservoir cells, because the final target groups use the aggregated **coal-oil** and **hydro dispatchable** categories.


### 1.2.1 Fossil coal-oil aggregated technology



In [ ]:
# =========================
# CELL – Per-Tech MLP for Fossil Coal+Oil (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_fossil_coal_oil_mw"   # aggregated target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features separate
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window train/val
# ---------------------------
X_train_flat = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")).values
Y_train_flat = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")).values
X_val_flat   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")).values
Y_val_flat   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, 24, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, 24, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# Extract target only
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:   {Y_val.shape}")

# ---------------------------
# 3) MLP model
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.0011
HIDDEN      = [50, 40]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Best weights restored.")

    model_path = os.path.join(TECH_DIR, "best_mlp_coal_oil.keras")
    model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_coal_oil.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled val+test and strict day-ahead forecast
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

full_df = pd.concat([val_scaled, test_scaled], axis=0)

test_days = sorted(test_scaled.index.normalize().unique())

print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    H = full_df.loc[hist_start:hist_end][hist_feature_cols]
    F = full_df.loc[horizon_start:horizon_end][da_feature_cols]

    if len(H) != 24 or len(F) != 24:
        continue

    X_day = np.concatenate([H.values, F.values], axis=1).reshape(1, 24, F_total)

    y_pred_scaled = model.predict(X_day, verbose=0)[0]
    y_true_scaled = full_df.loc[horizon_start:horizon_end][TECH_NAME].values

    all_pred_scaled.append(y_pred_scaled)
    all_true_scaled.append(y_true_scaled)
    all_time_index.append(full_df.loc[horizon_start:horizon_end].index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
try:
    scaler_y
except:
    raise RuntimeError("scaler_y missing. Run prep cell first.")

n_techs = len(gen_cols)
k       = tech_idx
T       = len(y_true_scaled)

y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))
y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(y_true_scaled_full)[:, k]
y_pred = scaler_y.inverse_transform(y_pred_scaled_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_h = np.sqrt(mean_squared_error(y_true, y_pred))
mae_h  = mean_absolute_error(y_true, y_pred)
eps = 1e-6
mape_h = np.mean(np.abs((y_true - y_pred) /
                        np.clip(np.abs(y_true), eps, None))) * 100
nmae_h = mae_h / (np.mean(y_true) + eps)

print("\n[METRICS – HOURLY 2024, FOSSIL COAL+OIL]")
print(f"RMSE      = {rmse_h:.4f} MW")
print(f"MAE       = {mae_h:.4f} MW")
print(f"MAPE      = {mape_h:.2f} %")
print(f"nMAE_mean = {nmae_h:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (np.mean(y_true) + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "y_true_mw": "mean",
    "abs_error": "mean",
    "y_pred_mw": (lambda s: np.sqrt(np.mean((s - hourly_df.loc[s.index, "y_true_mw"])**2)))
})

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_df.reset_index().to_csv(os.path.join(TECH_DIR, "coal_oil_hourly_predictions_2024.csv"), index=False)
daily_df.to_csv(os.path.join(TECH_DIR, "coal_oil_daily_metrics_2024.csv"))
pd.DataFrame([{
    "RMSE_MW": rmse_h,
    "MAE_MW": mae_h,
    "MAPE_percent": mape_h,
    "nMAE_mean": nmae_h
}]).to_csv(os.path.join(TECH_DIR, "coal_oil_overall_metrics_2024.csv"), index=False)

# ---------------------------
# 10) Plots
# ---------------------------

# Actual vs predicted
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Fossil Coal+Oil – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "coal_oil_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

# Normalized abs error histogram
plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Fossil Coal+Oil – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "coal_oil_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for fossil coal+oil finished.")


### 1.2.2 Biomass


In [ ]:
# =========================
# CELL – Per-Tech MLP for Biomass (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_biomass_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and biomass index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract biomass only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
    best_model = model  # use this directly

    # Save model in native Keras format in tech subfolder (optional, no reload here)
    model_path = os.path.join(TECH_DIR, "best_mlp_biomass.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_biomass.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for biomass (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, BIOMASS]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, BIOMASS]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path = os.path.join(TECH_DIR, "biomass_hourly_predictions_2024.csv")
daily_csv_path  = os.path.join(TECH_DIR, "biomass_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "biomass_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Biomass Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "biomass_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Biomass – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "biomass_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for biomass finished.")


### 1.2.3 Wind onshore


In [ ]:
# =========================
# CELL – Per-Tech MLP for Wind Onshore (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_wind_onshore_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract wind onshore only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
    best_model = model

    # Save model in native Keras format in tech subfolder (optional, no reload here)
    model_path = os.path.join(TECH_DIR, "best_mlp_wind_onshore.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_wind_onshore.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for wind onshore (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, WIND_ONSHORE]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, WIND_ONSHORE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "wind_onshore_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "wind_onshore_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "wind_onshore_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Wind Onshore Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "wind_onshore_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Wind Onshore – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "wind_onshore_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for wind onshore finished.")


### 1.2.4 Fossil gas


In [ ]:
# =========================
# CELL – Per-Tech MLP for Fossil Gas (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_fossil_gas_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols.")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")

# Hist = all gen + calendar
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN/VAL datasets
# ---------------------------
X_train = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")).values
Y_train = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")).values
X_val   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")).values
Y_val   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")).values

n_train = X_train.shape[0]
n_val   = X_val.shape[0]
F_total = X_train.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

# reshape
X_train_seq = X_train.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# select fossil gas only
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:   {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H)  # 24h output
])

model.compile(
    loss=LOSS_FUNC,
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RT),
    metrics=["mae"]
)

print(model.summary())

es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True
)

# ---------------------------
# 4) Train
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        validation_data=(X_val_seq, Y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    model_path = os.path.join(TECH_DIR, "best_mlp_fossil_gas.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_fossil_gas.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load scaled VAL + TEST for strict 24→24 forecasting
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc")

full_df = pd.concat([val_scaled, test_scaled]).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_times = []

for day in test_days:
    h0 = pd.Timestamp(day)
    h1 = h0 + timedelta(hours=23)

    hist = full_df.loc[h0 - timedelta(hours=24):h0 - timedelta(hours=1)]
    horiz = full_df.loc[h0:h1]

    if len(hist) != 24 or len(horiz) != 24:
        continue

    H = hist[hist_feature_cols].values
    F = horiz[da_feature_cols].values
    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_s = best_model.predict(X_day, verbose=0)[0]
    y_true_s = horiz[TECH_NAME].values

    all_pred_scaled.append(y_pred_s)
    all_true_scaled.append(y_true_s)
    all_times.append(horiz.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_times)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("Run prep cell first to load scaler_y.")

T = len(y_true_scaled)
n_techs = len(gen_cols)
k       = tech_idx

Y_true_full = np.zeros((T, n_techs))
Y_pred_full = np.zeros((T, n_techs))

Y_true_full[:, k] = y_true_scaled
Y_pred_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(Y_true_full)[:, k]
y_pred = scaler_y.inverse_transform(Y_pred_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
eps  = 1e-6
mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100
mean_true = np.mean(y_true)
nmae = mae / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, FOSSIL GAS]")
print(f"RMSE      = {rmse:.3f} MW")
print(f"MAE       = {mae:.3f} MW")
print(f"MAPE      = {mape:.2f} %")
print(f"nMAE_mean = {nmae:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true": y_true,
    "y_pred": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_err"] = np.abs(hourly_df["y_true"] - hourly_df["y_pred"])
hourly_df["nabs_err_mean"] = hourly_df["abs_err"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).apply(
    lambda g: pd.Series({
        "RMSE_MW": np.sqrt(np.mean((g.y_true - g.y_pred)**2)),
        "MAE_MW": np.mean(np.abs(g.y_true - g.y_pred)),
        "MAPE_percent": np.mean(
            np.abs((g.y_true - g.y_pred) /
                   np.clip(np.abs(g.y_true), eps, None))
        ) * 100,
        "nMAE_mean": np.mean(np.abs(g.y_true - g.y_pred)) / (np.mean(g.y_true) + eps),
        "mean_true_MW": np.mean(g.y_true)
    })
)

print("\n[METRICS – DAILY 2024, FOSSIL GAS]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save results
# ---------------------------
hourly_df.to_csv(os.path.join(TECH_DIR, "fossil_gas_hourly_predictions_2024.csv"))
daily_df.to_csv(os.path.join(TECH_DIR, "fossil_gas_daily_metrics_2024.csv"))

pd.DataFrame([{
    "RMSE_MW": rmse,
    "MAE_MW": mae,
    "MAPE_percent": mape,
    "nMAE_mean": nmae,
    "mean_true_MW": mean_true
}]).to_csv(os.path.join(TECH_DIR, "fossil_gas_overall_metrics_2024.csv"), index=False)

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df.y_true, label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df.y_pred, label="Predicted", linewidth=0.8)
plt.title("Fossil Gas – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "fossil_gas_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_err_mean"], bins=50)
plt.title("Fossil Gas – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "fossil_gas_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for fossil gas finished.")


### 1.2.5 Nuclear


In [ ]:
# =========================
# CELL – Per-Tech MLP for Nuclear (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_nuclear_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [40, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

# EarlyStopping
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    # Save
    model_path = os.path.join(TECH_DIR, "best_mlp_nuclear.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_nuclear.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Strict day-ahead prediction on 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_slice    = full_df.loc[horizon_start - timedelta(hours=24):horizon_start - timedelta(hours=1)]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != 24 or len(horizon_slice) != 24:
        print(f"[WARN] Skipping {day}")
        continue

    H = hist_slice[hist_feature_cols].values
    F = horizon_slice[da_feature_cols].values

    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]
    y_true_day_scaled = horizon_slice[TECH_NAME].values

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
n_techs = len(gen_cols)
k       = tech_idx
T       = len(y_true_scaled)

y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, NUCLEAR]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "sq_error": lambda x: np.sqrt(np.mean(x)),
    "abs_error": "mean",
    "y_true_mw": "mean"
})

daily_df.rename(columns={
    "sq_error": "RMSE_MW",
    "abs_error": "MAE_MW",
    "y_true_mw": "mean_true_MW"
}, inplace=True)

daily_df["MAPE_percent"] = (daily_df["MAE_MW"] / np.clip(daily_df["mean_true_MW"], eps, None)) * 100
daily_df["nMAE_mean"]    = daily_df["MAE_MW"] / (daily_df["mean_true_MW"] + eps)

print("\n[METRICS – DAILY 2024, NUCLEAR]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "nuclear_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "nuclear_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "nuclear_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}]).to_csv(summary_csv_path, index=False)

print(f"[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Nuclear – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "nuclear_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Nuclear – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "nuclear_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for nuclear finished.")


### 1.2.6 Solar


In [ ]:
# =========================
# CELL – Per-Tech MLP for Solar (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_solar_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total       = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract solar only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    # Save model in native Keras format in tech subfolder
    model_path = os.path.join(TECH_DIR, "best_mlp_solar.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_solar.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                   # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23) # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total)
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for solar (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]
y_pred = np.maximum(y_pred, 0.0)
# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, SOLAR]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) /
    np.abs(hourly_df["y_true_mw"]),
    np.nan
)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom  = np.clip(np.abs(y_t), eps, None)
    mape_d = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, SOLAR]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "solar_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "solar_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "solar_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------

# (a) Hourly actual vs predicted
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted",
         linewidth=0.8, alpha=0.7)
plt.title("Solar – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "solar_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Solar – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "solar_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for solar finished.")


### 1.2.7 Waste


In [ ]:
# =========================
# CELL – Per-Tech MLP for Waste (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_waste_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract waste only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    # Save model in native Keras format in tech subfolder
    model_path = os.path.join(TECH_DIR, "best_mlp_waste.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_waste.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# history context = val tail + test
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping {day.date()} – hist={len(hist_slice)}, hor={len(horizon_slice)}")
        continue

    H = hist_slice[hist_feature_cols].values
    F = horizon_slice[da_feature_cols].values

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1).reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]   # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values           # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted for waste.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index,  axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for waste (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("scaler_y not found – run the prep cell in the same session first.")

n_techs = len(gen_cols)
k       = tech_idx

T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(
    np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))
) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, WASTE]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom  = np.clip(np.abs(y_t), eps, None)
    mape_d = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, WASTE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "waste_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "waste_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "waste_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------

# (a) Hourly actual vs predicted
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Waste – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "waste_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Waste – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "waste_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for waste finished.")


### 1.2.8 Other renewables


In [ ]:
# =========================
# CELL – Per-Tech MLP for Other Renewables (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_other_renewable_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists (gen/da/cal) and find index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total       = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, 24, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, 24, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# Extract target tech (other renewables)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [52, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H)
])

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    # Save model
    model_path = os.path.join(TECH_DIR, "best_mlp_other_renewables.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_other_renewables.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Strict day-ahead test forecasts (CarbonCast)
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                           parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                           parse_dates=["datetime_utc"]).set_index("datetime_utc")

full_df = pd.concat([val_scaled, test_scaled]).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    s0 = pd.Timestamp(day)
    s1 = s0 + timedelta(hours=23)

    hist_start = s0 - timedelta(hours=24)
    hist_end   = s0 - timedelta(hours=1)

    hist = full_df.loc[hist_start:hist_end]
    fut  = full_df.loc[s0:s1]

    if len(hist) != 24 or len(fut) != 24:
        continue

    H = hist[hist_feature_cols].values
    F = fut[da_feature_cols].values
    X = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    yp = best_model.predict(X, verbose=0)[0]
    yt = fut[TECH_NAME].values

    all_pred_scaled.append(yp)
    all_true_scaled.append(yt)
    all_time_index.append(fut.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print("[TEST] Total hourly predictions:", len(y_true_scaled))

# ---------------------------
# 6) Inverse scaling
# ---------------------------
n_techs = len(gen_cols)
k = tech_idx

T = len(y_true_scaled)
yt_full = np.zeros((T, n_techs))
yp_full = np.zeros((T, n_techs))
yt_full[:, k] = y_true_scaled
yp_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(yt_full)[:, k]
y_pred = scaler_y.inverse_transform(yp_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100
mean_true   = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, OTHER RENEWABLES]")
print(f"RMSE      = {rmse_hourly:.3f} MW")
print(f"MAE       = {mae_hourly:.3f} MW")
print(f"MAPE      = {mape_hourly:.3f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc")

daily_df = []
for d, g in hourly_df.groupby(hourly_df.index.normalize()):
    y_t = g["y_true_mw"].values
    y_p = g["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    mape_d = np.mean(np.abs((y_t - y_p) / np.clip(np.abs(y_t), eps, None))) * 100
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_df.append({
        "date": d.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_df).sort_values("date")

print("\n[METRICS – DAILY SUMMARY]")
print(daily_df.describe()[["RMSE_MW","MAE_MW","MAPE_percent","nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_path  = os.path.join(TECH_DIR, "other_renewables_hourly_predictions_2024.csv")
daily_path   = os.path.join(TECH_DIR, "other_renewables_daily_metrics_2024.csv")
summary_path = os.path.join(TECH_DIR, "other_renewables_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_path, index=False)
daily_df.to_csv(daily_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_path, index=False)

print(f"[SAVE] Hourly  -> {hourly_path}")
print(f"[SAVE] Daily   -> {daily_path}")
print(f"[SAVE] Summary -> {summary_path}")

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Other Renewables – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1 = os.path.join(TECH_DIR, "other_renewables_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1, dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / (mean_true + eps), bins=50)
plt.title("Other Renewables – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2 = os.path.join(TECH_DIR, "other_renewables_norm_abs_error_hist_2024.png")
plt.savefig(plot2, dpi=150)
plt.show()

print(f"[PLOT] Saved -> {plot1}")
print(f"[PLOT] Saved -> {plot2}")

print("\n[DONE] Per-tech MLP for other renewables finished.")


### 1.2.9 Hydro run-of-river


In [ ]:
# =========================
# CELL – Per-Tech MLP for Hydro Run-of-River (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_hydro_run_of_river_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists & tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape}")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape}")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:  {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H),
])

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(monitor="val_loss", mode="min", patience=10,
                   verbose=1, restore_best_weights=True)

# ---------------------------
# 4) Train
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model

    # Save model
    model_path = os.path.join(TECH_DIR, "best_mlp_hydro_run_of_river.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_hydro_run_of_river.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Strict day-ahead forecasting for 2024
# ---------------------------
val_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc")

print("[TEST] val_scaled:", val_scaled.shape,
      "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape,
      "|", test_scaled.index.min(), "->", test_scaled.index.max())

full_df = pd.concat([val_scaled, test_scaled]).sort_index()
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    H_slice = full_df.loc[hist_start:hist_end]
    F_slice = full_df.loc[horizon_start:horizon_end]

    if len(H_slice) != 24 or len(F_slice) != 24:
        print(f"[WARN] Skipping {day.date()} (incomplete window)")
        continue

    H = H_slice[hist_feature_cols].values
    F = F_slice[da_feature_cols].values

    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_scaled = best_model.predict(X_day, verbose=0)[0]
    y_true_scaled = F_slice[TECH_NAME].values

    all_pred_scaled.append(y_pred_scaled)
    all_true_scaled.append(y_true_scaled)
    all_time_index.append(F_slice.index.values)

# Stack predictions
y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling (requires scaler_y)
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("scaler_y missing — run prep cell first.")

n_techs = len(gen_cols)
k = tech_idx
T = len(y_true_scaled)

yt_full  = np.zeros((T, n_techs))
yp_full  = np.zeros((T, n_techs))

yt_full[:, k] = y_true_scaled
yp_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(yt_full)[:, k]
y_pred = scaler_y.inverse_transform(yp_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, HYDRO_RUN_OF_RIVER]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "sq_error":  lambda x: np.sqrt(np.mean(x)),
    "abs_error": "mean",
    "y_true_mw": "mean"
}).rename(columns={
    "sq_error": "RMSE_MW",
    "abs_error": "MAE_MW",
    "y_true_mw": "mean_true_MW"
})
daily_df["MAPE_percent"] = (
    daily_df["MAE_MW"] /
    np.clip(np.abs(daily_df["mean_true_MW"]), eps, None)
) * 100
daily_df["nMAE_mean"] = daily_df["MAE_MW"] / (daily_df["mean_true_MW"] + eps)

print("\n[METRICS – DAILY 2024, HYDRO_RUN_OF_RIVER]")
print(daily_df.describe()[["RMSE_MW","MAE_MW","MAPE_percent","nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_df.reset_index().to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_hourly_predictions_2024.csv"),
    index=False
)
daily_df.reset_index().to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_daily_metrics_2024.csv"),
    index=False
)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_overall_metrics_2024.csv"),
    index=False
)

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.7)
plt.title("Hydro Run-of-River – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "hydro_run_of_river_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Hydro Run-of-River – Normalized Abs Error Distribution (2024)")
plt.xlabel("|Error| / mean_true")
plt.ylabel("Frequency (hours)")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "hydro_run_of_river_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for hydro run-of-river finished.")


### 1.2.10 Other generation


In [ ]:
# =========================
# CELL – Per-Tech MLP for Other (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_other_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract "other" only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [48, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
    best_model = model  # use this directly

    # Save model in native Keras format in tech subfolder
    model_path = os.path.join(TECH_DIR, "best_mlp_other.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_other.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for OTHER (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, OTHER]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, OTHER]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "other_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "other_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "other_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Other Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "other_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Other – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "other_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for other finished.")


### 1.2.11 Hydro dispatchable


In [ ]:
# =========================
# CELL – Per-Tech MLP for Hydro Dispatchable (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = str(TIER1_WORK)
TECH_NAME     = "gen_hydro_dispatchable_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))


# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)



Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract hydro dispatchable only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [51, 36]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
if TRAIN_ANN:
    history = model.fit(
        X_train_seq, Y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_seq, Y_val),
        callbacks=[es],
        verbose=2
    )

    print("[TRAIN] Training finished. Best weights restored.")
    best_model = model  # use this directly

    # Save model in native Keras format in tech subfolder
    model_path = os.path.join(TECH_DIR, "best_mlp_hydro_dispatchable.keras")
    best_model.save(model_path)

else:
    model_path = os.path.join(TECH_DIR, "best_mlp_hydro_dispatchable.keras")
    archived = DATA / "results/ML/tier1" / TECH_ARCHIVES[TECH_NAME] / Path(model_path).name
    model = best_model = tf.keras.models.load_model(archived)
    print(f"[MODEL] Loaded supplied weights: {archived}")
print(f"[MODEL] Ready: {model_path if TRAIN_ANN else archived}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training





    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for hydro dispatchable (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, HYDRO_DISPATCHABLE]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, HYDRO_DISPATCHABLE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "hydro_dispatchable_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "hydro_dispatchable_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Hydro Dispatchable Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Hydro Dispatchable – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for hydro dispatchable finished.")
